# ERM vs EQRM Hyperparameter Sweep on Spawrious (DomainBed)

> **Runtime:** Google Colab Pro — A100 GPU  
> **Goal:** Deterministic sweep comparing ERM and EQRM across three Spawrious M2M difficulty levels, with checkpointed final weights for downstream mechanistic interpretability.  
>
> **Key design decision:** We bypass DomainBed's random-search sweep launcher and instead run a fixed hyperparameter loop so every run is deterministic and directly comparable.

## Cell 1 — Verify GPU & Mount Google Drive

In [ ]:
import torch, os

# ── GPU sanity check ──
assert torch.cuda.is_available(), "No GPU detected — switch to an A100 runtime."
gpu_name = torch.cuda.get_device_name(0)
print(f"GPU  : {gpu_name}")
print(f"CUDA : {torch.version.cuda}")
print(f"PyTorch : {torch.__version__}")

# ── Mount Google Drive for persistent storage ──
from google.colab import drive
drive.mount("/content/drive")

DRIVE_OUTPUT = "/content/drive/MyDrive/domainbed_spawrious_sweep"
os.makedirs(DRIVE_OUTPUT, exist_ok=True)
print(f"\nPersistent output dir: {DRIVE_OUTPUT}")

GPU  : NVIDIA A100-SXM4-40GB
CUDA : 12.8
PyTorch : 2.10.0+cu128
Mounted at /content/drive

Persistent output dir: /content/drive/MyDrive/domainbed_spawrious_sweep


## Cell 2 — Clone DomainBed & Install Spawrious

In [ ]:
import os, sys

DOMAINBED_DIR = "/content/DomainBed"

# Clone DomainBed (not pip-installable — no setup.py)
if not os.path.isdir(DOMAINBED_DIR):
    !git clone https://github.com/facebookresearch/DomainBed.git {DOMAINBED_DIR}
else:
    print(f"DomainBed already cloned at {DOMAINBED_DIR}")

# Add DomainBed to Python path so 'import domainbed' works
if DOMAINBED_DIR not in sys.path:
    sys.path.insert(0, DOMAINBED_DIR)

# Spawrious registers its datasets inside DomainBed's dataset registry
!pip install git+https://github.com/aengusl/spawrious.git --quiet

# Spawrious may need additional deps
!pip install wilds --quiet

print(f"\n✓ DomainBed cloned at: {DOMAINBED_DIR}")
print(f"✓ sys.path includes DomainBed")

Cloning into '/content/DomainBed'...
remote: Enumerating objects: 1358, done.
remote: Counting objects: 100% (78/78), done.
remote: Compressing objects: 100% (39/39), done.
remote: Total 1358 (delta 56), reused 39 (delta 39), pack-reused 1280 (from 2)
Receiving objects: 100% (1358/1358), 1.11 MiB | 8.50 MiB/s, done.
Resolving deltas: 100% (797/797), done.
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.2/126.2 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.8/78.8 kB 9.6 MB/s eta 0:00:00

✓ DomainBed cloned at: /content/DomainBed
✓ sys.path includes DomainBed


## Cell 3 — Verify Imports & Dataset Registry

In [ ]:
# Confirm DomainBed can see the Spawrious datasets and the two algorithms
import domainbed
from domainbed import datasets as db_datasets
from domainbed import algorithms as db_algorithms

REQUIRED_DATASETS = [
    "SpawriousM2M_easy",
    "SpawriousM2M_medium",
    "SpawriousM2M_hard",
]
REQUIRED_ALGOS = ["ERM", "EQRM"]

print("── Dataset check ──")
for name in REQUIRED_DATASETS:
    cls = vars(db_datasets).get(name)
    assert cls is not None, f"{name} not found in DomainBed dataset registry!"
    print(f"  ✓ {name:30s}  environments: {cls.ENVIRONMENTS}")

print("\n── Algorithm check ──")
for name in REQUIRED_ALGOS:
    cls = vars(db_algorithms).get(name)
    assert cls is not None, f"{name} not found in DomainBed algorithm registry!"
    print(f"  ✓ {name}")

print("\n── DomainBed location ──")
print(f"  {domainbed.__file__}")

── Dataset check ──
  ✓ SpawriousM2M_easy               environments: ['Test', 'SC_group_1', 'SC_group_2']
  ✓ SpawriousM2M_medium             environments: ['Test', 'SC_group_1', 'SC_group_2']
  ✓ SpawriousM2M_hard               environments: ['Test', 'SC_group_1', 'SC_group_2']

── Algorithm check ──
  ✓ ERM
  ✓ EQRM

── DomainBed location ──
  /content/DomainBed/domainbed/__init__.py


/content/DomainBed/domainbed/lib/misc.py:30: SyntaxWarning: invalid escape sequence '\d'
  ''' return proj_{B(h, \delta)}(adv_h), Euclidean projection to Euclidean ball'''


## Cell 4 — Locate the DomainBed `train.py` Entry Point

In [ ]:
import pathlib

TRAIN_SCRIPT = pathlib.Path(DOMAINBED_DIR) / "domainbed" / "scripts" / "train.py"

assert TRAIN_SCRIPT.exists(), (
    f"Cannot find train.py at {TRAIN_SCRIPT}.\n"
    f"Contents of {DOMAINBED_DIR}/domainbed/scripts/:\n"
    + str(list((pathlib.Path(DOMAINBED_DIR) / 'domainbed' / 'scripts').glob('*')))
)
print(f"train.py found at:\n  {TRAIN_SCRIPT}")

train.py found at:
  /content/DomainBed/domainbed/scripts/train.py


## Cell 5 — Define the Sweep Configuration

This is the core of the experiment. We explicitly set every hyperparameter instead
of relying on DomainBed's random search so the comparison is fully controlled.

In [ ]:
import json, itertools

# ═══════════════════════════════════════════════════════════════════
# DATASETS — Spawrious M2M at three difficulty levels
# The benchmark contains 4 dog breeds (Bulldog, Dachshund, Corgi,
# Labrador) with 6 background locations. In the M2M splits, *groups*
# of classes correlate with *groups* of backgrounds, and these
# correlations are reversed in the test environment.
# ═══════════════════════════════════════════════════════════════════
DATASETS = [
    "SpawriousM2M_easy",
    "SpawriousM2M_medium",
    "SpawriousM2M_hard",
]

# ═══════════════════════════════════════════════════════════════════
# ALGORITHMS
#   ERM  — standard empirical risk minimization (baseline)
#   EQRM — empirical quantile risk minimization (targets the
#           worst-quantile loss across environments)
# ═══════════════════════════════════════════════════════════════════
ALGORITHMS = ["ERM", "EQRM"]

# ═══════════════════════════════════════════════════════════════════
# TRAINING KNOBS
# ═══════════════════════════════════════════════════════════════════
TEST_ENV       = 0       # held-out test environment index
STEPS          = 5001    # DomainBed default (N_STEPS in datasets.py)
CKPT_FREQ      = 5001    # checkpoint at the final step

# ═══════════════════════════════════════════════════════════════════
# HYPERPARAMETERS (passed via --hparams JSON string)
#
# Shared across both algorithms:
#   • resnet18        = true   (matches Spawrious paper Table 3)
#   • batch_size      = 256    (maximize A100 throughput; paper used 128)
#   • resnet_dropout  = 0.1    (paper Sec 5: "We keep dropout (0.1) fixed")
#
# EQRM-specific (Eastwood et al., NeurIPS 2022, Appendix E.3):
#   • eqrm_quantile      = 0.75  (paper default; swept U(0.5, 0.99))
#   • eqrm_burnin_iters  = 2500  (paper default; EQRM runs as plain
#                                  ERM for the first 2500 steps, then
#                                  activates the quantile penalty for
#                                  the remaining 2501 steps.
#                                  NOTE: if you ever drop --steps below
#                                  2500, you MUST lower this too or EQRM
#                                  silently degrades to plain ERM.)
#   • eqrm_lr             = 1e-6  (paper default; lower LR post burn-in
#                                  to stabilize quantile optimization)
# ═══════════════════════════════════════════════════════════════════
SHARED_HPARAMS = {
    "resnet18":       True,
    "batch_size":     128,
    "resnet_dropout": 0.1,
}

ALGO_SPECIFIC_HPARAMS = {
    "ERM":  {},
    "EQRM": {
        "eqrm_quantile":     0.75,
        "eqrm_burnin_iters": 2500,
        "eqrm_lr":           1e-6,
    },
}

# ── Build the full (dataset × algorithm) run manifest ──
runs = []
for ds, algo in itertools.product(DATASETS, ALGORITHMS):
    hparams = {**SHARED_HPARAMS, **ALGO_SPECIFIC_HPARAMS[algo]}
    runs.append({"dataset": ds, "algorithm": algo, "hparams": hparams})

print(f"Total runs: {len(runs)}")
print(f"  {len(DATASETS)} datasets × {len(ALGORITHMS)} algorithms\n")
for i, r in enumerate(runs):
    print(f"  [{i}] {r['dataset']:25s}  {r['algorithm']:5s}  "
          f"hparams={json.dumps(r['hparams'])}")

Total runs: 6
  3 datasets × 2 algorithms

  [0] SpawriousM2M_easy          ERM    hparams={"resnet18": true, "batch_size": 128, "resnet_dropout": 0.1}
  [1] SpawriousM2M_easy          EQRM   hparams={"resnet18": true, "batch_size": 128, "resnet_dropout": 0.1, "eqrm_quantile": 0.75, "eqrm_burnin_iters": 2500, "eqrm_lr": 1e-06}
  [2] SpawriousM2M_medium        ERM    hparams={"resnet18": true, "batch_size": 128, "resnet_dropout": 0.1}
  [3] SpawriousM2M_medium        EQRM   hparams={"resnet18": true, "batch_size": 128, "resnet_dropout": 0.1, "eqrm_quantile": 0.75, "eqrm_burnin_iters": 2500, "eqrm_lr": 1e-06}
  [4] SpawriousM2M_hard          ERM    hparams={"resnet18": true, "batch_size": 128, "resnet_dropout": 0.1}
  [5] SpawriousM2M_hard          EQRM   hparams={"resnet18": true, "batch_size": 128, "resnet_dropout": 0.1, "eqrm_quantile": 0.75, "eqrm_burnin_iters": 2500, "eqrm_lr": 1e-06}


## Cell 6 — Create Working Directories & Download Spawrious Data

In [ ]:
import os

# Local working directory (fast SSD inside the Colab VM)
LOCAL_DATA_DIR   = "/content/domainbed_data"
LOCAL_OUTPUT_DIR = "/content/domainbed_outputs"

os.makedirs(LOCAL_DATA_DIR, exist_ok=True)
os.makedirs(LOCAL_OUTPUT_DIR, exist_ok=True)

print(f"Dataset cache  : {LOCAL_DATA_DIR}")
print(f"Local outputs  : {LOCAL_OUTPUT_DIR}")
print(f"Drive mirror   : {DRIVE_OUTPUT}")

# ── Download Spawrious images ──
# DomainBed's Spawrious dataset classes expect images on disk at
# <data_dir>/<type_index>/<location>/<breed>/*.jpg
# The spawrious package downloads them from HuggingFace on first call.
# We trigger the download here so train.py doesn't fail.
#
# NOTE: get_spawrious_dataset() downloads + extracts the images, then
# tries to build a timm model which can crash (AttributeError in timm).
# We only need the files on disk, so we catch and ignore that error.
from spawrious.torch import get_spawrious_dataset

print("\nDownloading Spawrious M2M images (first time takes ~5-10 min)...")
try:
    _ = get_spawrious_dataset(dataset_name="m2m_hard", root_dir=LOCAL_DATA_DIR)
except (AttributeError, Exception) as e:
    # The download/extract succeeded but model init failed — that's fine,
    # we only need the image files, not the spawrious model wrapper.
    print(f"  (Ignored post-download error: {type(e).__name__}: {e})")
print("✓ Spawrious data ready")

# The spawrious package extracts images into a 'spawrious224' subfolder.
# DomainBed expects data_dir to point directly to the folder containing
# the <type_index>/<location>/<breed>/ tree.
SPAWRIOUS_ROOT = os.path.join(LOCAL_DATA_DIR, "spawrious224")
if not os.path.isdir(SPAWRIOUS_ROOT):
    # Fallback: maybe images are directly in LOCAL_DATA_DIR
    SPAWRIOUS_ROOT = LOCAL_DATA_DIR

# Verify the expected folder structure exists
expected = os.path.join(SPAWRIOUS_ROOT, "0", "desert", "bulldog")
assert os.path.isdir(expected), (
    f"Expected folder not found: {expected}\n"
    f"Contents of {LOCAL_DATA_DIR}: {os.listdir(LOCAL_DATA_DIR)}"
)
n_images = len([f for f in os.listdir(expected) if f.endswith(('.jpg','.png','.jpeg'))])
print(f"  Data root: {SPAWRIOUS_ROOT}")
print(f"  Sample check: {expected} → {n_images} images")

Dataset cache  : /content/domainbed_data
Local outputs  : /content/domainbed_outputs
Drive mirror   : /content/drive/MyDrive/domainbed_spawrious_sweep

Dataset not found. Downloading...


100%|██████████| 15.8G/15.8G [02:54<00:00, 90.2MiB/s]


Dataset downloaded. Extracting...
Extracting dataset...
Dataset extracted. Delete tar file.
  (Ignored post-download error: AttributeError: 'NoneType' object has no attribute 'startswith')
✓ Spawrious data ready
  Data root: /content/domainbed_data/spawrious224
  Sample check: /content/domainbed_data/spawrious224/0/desert/bulldog → 3168 images


## Cell 7 — Execute the Sweep

Each iteration:
1. Launches `train.py` with our fixed hparams.
2. Streams stdout/stderr live so you can monitor loss curves.
3. Copies the full output tree to Google Drive after each run finishes (crash-safe — you never lose a completed run).

**Estimated time:** ~15–25 min per run on A100 × 6 runs ≈ 1.5–2.5 hours total.

In [ ]:
import subprocess, shutil, json, time, os

# Ensure subprocesses can also import domainbed
env = os.environ.copy()
env["PYTHONPATH"] = DOMAINBED_DIR + ":" + env.get("PYTHONPATH", "")

for idx, run in enumerate(runs):
    ds   = run["dataset"]
    algo = run["algorithm"]
    hp   = json.dumps(run["hparams"])

    # Per-run output directory
    run_tag    = f"{ds}__{algo}"
    run_outdir = os.path.join(LOCAL_OUTPUT_DIR, run_tag)
    os.makedirs(run_outdir, exist_ok=True)

    print("\n" + "=" * 72)
    print(f"  RUN {idx + 1}/{len(runs)}:  {ds}  ×  {algo}")
    print(f"  hparams : {hp}")
    print(f"  output  : {run_outdir}")
    print("=" * 72 + "\n")

    cmd = [
        "python", str(TRAIN_SCRIPT),
        "--data_dir",                    SPAWRIOUS_ROOT,
        "--output_dir",                  run_outdir,
        "--dataset",                     ds,
        "--algorithm",                   algo,
        "--test_env",                    str(TEST_ENV),
        "--steps",                       str(STEPS),
        "--checkpoint_freq",             str(CKPT_FREQ),
        "--save_model_every_checkpoint",
        "--hparams",                     hp,
    ]

    t0 = time.time()

    # Stream output line-by-line for live Colab monitoring
    proc = subprocess.Popen(
        cmd,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in proc.stdout:
        print(line, end="")
    proc.wait()

    elapsed = time.time() - t0
    minutes = elapsed / 60

    if proc.returncode != 0:
        print(f"\n✗ FAILED  ({algo} on {ds})  — return code {proc.returncode}")
        print("  Continuing to next run...\n")
        continue

    print(f"\n✓ DONE  {algo} on {ds}  ({minutes:.1f} min)")

    # ── Persist to Google Drive ──
    drive_dest = os.path.join(DRIVE_OUTPUT, run_tag)
    if os.path.exists(drive_dest):
        shutil.rmtree(drive_dest)
    shutil.copytree(run_outdir, drive_dest)
    print(f"  → Copied to Drive: {drive_dest}")

print("\n" + "=" * 72)
print("  ALL RUNS COMPLETE")
print("=" * 72)


  RUN 1/6:  SpawriousM2M_easy  ×  ERM
  hparams : {"resnet18": true, "batch_size": 128, "resnet_dropout": 0.1}
  output  : /content/domainbed_outputs/SpawriousM2M_easy__ERM

Environment:
	Python: 3.12.13
	PyTorch: 2.10.0+cu128
	Torchvision: 0.25.0+cu128
	CUDA: 12.8
	CUDNN: 91002
	NumPy: 2.0.2
	PIL: 11.3.0
Args:
	algorithm: ERM
	checkpoint_freq: 5001
	data_dir: /content/domainbed_data/spawrious224
	dataset: SpawriousM2M_easy
	holdout_fraction: 0.2
	hparams: {"resnet18": true, "batch_size": 128, "resnet_dropout": 0.1}
	hparams_seed: 0
	output_dir: /content/domainbed_outputs/SpawriousM2M_easy__ERM
	save_model_every_checkpoint: True
	seed: 0
	skip_model_save: False
	steps: 5001
	task: domain_generalization
	test_envs: [0]
	trial_seed: 0
	uda_holdout_fraction: 0
HParams:
	batch_size: 128
	class_balanced: False
	data_augmentation: True
	dinov2: False
	freeze_bn: False
	lars: False
	linear_steps: 500
	lr: 5e-05
	nonlinear_classifier: False
	resnet18: True
	resnet50_augmix: True
	resnet_dropo

## Cell 8 — Collect & Display Results

DomainBed writes a `results.jsonl` file in each run's output directory.
This cell parses every run and builds a summary table.

In [ ]:
import json, os, glob
import pandas as pd

rows = []
pattern = os.path.join(LOCAL_OUTPUT_DIR, "*", "results.jsonl")

for results_file in sorted(glob.glob(pattern)):
    run_dir  = os.path.dirname(results_file)
    run_tag  = os.path.basename(run_dir)
    parts    = run_tag.split("__")
    dataset  = parts[0] if len(parts) >= 1 else run_tag
    algo     = parts[1] if len(parts) >= 2 else "?"

    # Read the last line (final checkpoint results)
    with open(results_file, "r") as f:
        lines = [l.strip() for l in f if l.strip()]
    if not lines:
        continue
    final = json.loads(lines[-1])

    row = {
        "dataset":   dataset,
        "algorithm": algo,
        "step":      final.get("step", "?"),
    }

    # Grab all accuracy keys
    for k, v in final.items():
        if "acc" in k:
            row[k] = round(v * 100, 2) if isinstance(v, (int, float)) else v

    rows.append(row)

if rows:
    df = pd.DataFrame(rows)
    fixed_cols = ["dataset", "algorithm", "step"]
    acc_cols   = sorted([c for c in df.columns if c not in fixed_cols])
    df = df[fixed_cols + acc_cols]

    print("═" * 80)
    print("  RESULTS SUMMARY  (accuracies in %)")
    print("═" * 80)
    print(df.to_string(index=False))
    print()
else:
    print("No results.jsonl files found. Run Cell 7 first.")

════════════════════════════════════════════════════════════════════════════════
  RESULTS SUMMARY  (accuracies in %)
════════════════════════════════════════════════════════════════════════════════
            dataset algorithm  step  env0_in_acc  env0_out_acc  env1_in_acc  env1_out_acc  env2_in_acc  env2_out_acc
  SpawriousM2M_easy      EQRM  5000        82.12         82.79       100.00         99.80       100.00         99.33
  SpawriousM2M_easy       ERM  5000        79.42         80.70       100.00         99.68       100.00         99.45
  SpawriousM2M_hard      EQRM  5000        58.73         59.12       100.00         99.57       100.00         99.88
  SpawriousM2M_hard       ERM  5000        53.01         54.22        99.95         99.41        99.87         99.41
SpawriousM2M_medium      EQRM  5000        57.88         57.73        99.99         99.25        99.99         99.37
SpawriousM2M_medium       ERM  5000        61.71         61.37        99.99         98.90        99

## Cell 9 — Verify Saved Checkpoints

Confirm that the final model weights exist for each run (needed for
mechanistic interpretability downstream).

In [ ]:
import os, glob

print("Checkpoint inventory:\n")
for run_dir in sorted(glob.glob(os.path.join(LOCAL_OUTPUT_DIR, "*"))):
    run_tag = os.path.basename(run_dir)
    ckpts   = sorted(glob.glob(os.path.join(run_dir, "**", "*.pkl"), recursive=True))
    print(f"  {run_tag}:")
    if ckpts:
        for c in ckpts:
            size_mb = os.path.getsize(c) / 1e6
            print(f"    {os.path.relpath(c, run_dir):45s}  {size_mb:7.1f} MB")
    else:
        print(f"    (no checkpoints found)")
    print()

# Also confirm Drive copies
print(f"\nDrive mirror ({DRIVE_OUTPUT}):")
if os.path.isdir(DRIVE_OUTPUT):
    for entry in sorted(os.listdir(DRIVE_OUTPUT)):
        full = os.path.join(DRIVE_OUTPUT, entry)
        if os.path.isdir(full):
            n_files = sum(len(files) for _, _, files in os.walk(full))
            print(f"  📁 {entry}/  ({n_files} files)")
        else:
            size_mb = os.path.getsize(full) / 1e6
            print(f"  📄 {entry}  ({size_mb:.1f} MB)")
else:
    print("  (not found)")

Checkpoint inventory:

  SpawriousM2M_easy__EQRM:
    model.pkl                                         94.4 MB
    model_step0.pkl                                   94.5 MB
    model_step5000.pkl                                94.5 MB

  SpawriousM2M_easy__ERM:
    model.pkl                                         94.4 MB
    model_step0.pkl                                   94.5 MB
    model_step5000.pkl                                94.5 MB

  SpawriousM2M_hard__EQRM:
    model.pkl                                         94.4 MB
    model_step0.pkl                                   94.5 MB
    model_step5000.pkl                                94.5 MB

  SpawriousM2M_hard__ERM:
    model.pkl                                         94.4 MB
    model_step0.pkl                                   94.5 MB
    model_step5000.pkl                                94.5 MB

  SpawriousM2M_medium__EQRM:
    model.pkl                                         94.4 MB
    model_step0.pkl             

## Appendix — Configuration Quick-Reference

| Parameter | Value | Rationale |
|---|---|---|
| `resnet18` | `true` | Matches Spawrious paper Table 3 (ResNet18 block) |
| `batch_size` | `128` | Maximize A100 throughput; Spawrious paper used 128 |
| `resnet_dropout` | `0.1` | Spawrious paper Sec 5: "We keep the dropout rate (0.1) … fixed" |
| `eqrm_quantile` | `0.75` | EQRM paper default (Eastwood et al., Appendix E.3); swept U(0.5, 0.99) |
| `eqrm_burnin_iters` | `2500` | EQRM paper default. EQRM runs as plain ERM for the first 2500 steps. If you set steps < 2500, you MUST lower this or EQRM never activates. |
| `eqrm_lr` | `1e-6` | EQRM paper default. Lower learning rate post burn-in stabilizes quantile optimization. |
| `--steps` | `5001` | DomainBed default (`N_STEPS` in `datasets.py`); matches Spawrious paper methodology |
| `--test_env` | `0` | Held-out environment 0 |
| `--checkpoint_freq` | `5001` | Save only the final weights |
| `--save_model_every_checkpoint` | flag | Ensures `.pkl` is written at checkpoint step |

In [ ]:
import json, itertools

# ═══════════════════════════════════════════════════════════════════
# DATASETS — Spawrious M2M at three difficulty levels
# The benchmark contains 4 dog breeds (Bulldog, Dachshund, Corgi,
# Labrador) with 6 background locations. In the M2M splits, *groups*
# of classes correlate with *groups* of backgrounds, and these
# correlations are reversed in the test environment.
# ═══════════════════════════════════════════════════════════════════
DATASETS = [
    "SpawriousM2M_easy",
    "SpawriousM2M_medium",
    "SpawriousM2M_hard",
]

# ═══════════════════════════════════════════════════════════════════
# ALGORITHMS
#   ERM  — standard empirical risk minimization (baseline)
#   EQRM — empirical quantile risk minimization (targets the
#           worst-quantile loss across environments)
# ═══════════════════════════════════════════════════════════════════
ALGORITHMS = ["EQRM"]

# ═══════════════════════════════════════════════════════════════════
# TRAINING KNOBS
# ═══════════════════════════════════════════════════════════════════
TEST_ENV       = 0       # held-out test environment index
STEPS          = 5001    # DomainBed default (N_STEPS in datasets.py)
CKPT_FREQ      = 5001    # checkpoint at the final step

# ═══════════════════════════════════════════════════════════════════
# HYPERPARAMETERS (passed via --hparams JSON string)
#
# Shared across both algorithms:
#   • resnet18        = true   (matches Spawrious paper Table 3)
#   • batch_size      = 256    (maximize A100 throughput; paper used 128)
#   • resnet_dropout  = 0.1    (paper Sec 5: "We keep dropout (0.1) fixed")
#
# EQRM-specific (Eastwood et al., NeurIPS 2022, Appendix E.3):
#   • eqrm_quantile      = 0.75  (paper default; swept U(0.5, 0.99))
#   • eqrm_burnin_iters  = 2500  (paper default; EQRM runs as plain
#                                  ERM for the first 2500 steps, then
#                                  activates the quantile penalty for
#                                  the remaining 2501 steps.
#                                  NOTE: if you ever drop --steps below
#                                  2500, you MUST lower this too or EQRM
#                                  silently degrades to plain ERM.)
#   • eqrm_lr             = 1e-6  (paper default; lower LR post burn-in
#                                  to stabilize quantile optimization)
# ═══════════════════════════════════════════════════════════════════
SHARED_HPARAMS = {
    "resnet18":       True,
    "batch_size":     128,
    "resnet_dropout": 0.1,
}

ALGO_SPECIFIC_HPARAMS = {
    "ERM":  {},
    "EQRM": {
        "eqrm_quantile":     0.75,
        "eqrm_burnin_iters": 2500,
        "eqrm_lr":           5e-5,
    },
}

# ── Build the full (dataset × algorithm) run manifest ──
runs = []
for ds, algo in itertools.product(DATASETS, ALGORITHMS):
    hparams = {**SHARED_HPARAMS, **ALGO_SPECIFIC_HPARAMS[algo]}
    runs.append({"dataset": ds, "algorithm": algo, "hparams": hparams})

print(f"Total runs: {len(runs)}")
print(f"  {len(DATASETS)} datasets × {len(ALGORITHMS)} algorithms\n")
for i, r in enumerate(runs):
    print(f"  [{i}] {r['dataset']:25s}  {r['algorithm']:5s}  "
          f"hparams={json.dumps(r['hparams'])}")

Total runs: 3
  3 datasets × 1 algorithms

  [0] SpawriousM2M_easy          EQRM   hparams={"resnet18": true, "batch_size": 128, "resnet_dropout": 0.1, "eqrm_quantile": 0.75, "eqrm_burnin_iters": 2500, "eqrm_lr": 5e-05}
  [1] SpawriousM2M_medium        EQRM   hparams={"resnet18": true, "batch_size": 128, "resnet_dropout": 0.1, "eqrm_quantile": 0.75, "eqrm_burnin_iters": 2500, "eqrm_lr": 5e-05}
  [2] SpawriousM2M_hard          EQRM   hparams={"resnet18": true, "batch_size": 128, "resnet_dropout": 0.1, "eqrm_quantile": 0.75, "eqrm_burnin_iters": 2500, "eqrm_lr": 5e-05}


In [ ]:
import subprocess, shutil, json, time, os

# Ensure subprocesses can also import domainbed
env = os.environ.copy()
env["PYTHONPATH"] = DOMAINBED_DIR + ":" + env.get("PYTHONPATH", "")

for idx, run in enumerate(runs):
    ds   = run["dataset"]
    algo = run["algorithm"]
    hp   = json.dumps(run["hparams"])

    # Per-run output directory
    # Per-run output directory
    lr_label = run["hparams"].get("eqrm_lr", "default")
    run_tag    = f"{ds}__{algo}__lr{lr_label}"
    run_outdir = os.path.join(LOCAL_OUTPUT_DIR, run_tag)
    os.makedirs(run_outdir, exist_ok=True)

    print("\n" + "=" * 72)
    print(f"  RUN {idx + 1}/{len(runs)}:  {ds}  ×  {algo}")
    print(f"  hparams : {hp}")
    print(f"  output  : {run_outdir}")
    print("=" * 72 + "\n")

    cmd = [
        "python", str(TRAIN_SCRIPT),
        "--data_dir",                    SPAWRIOUS_ROOT,
        "--output_dir",                  run_outdir,
        "--dataset",                     ds,
        "--algorithm",                   algo,
        "--test_env",                    str(TEST_ENV),
        "--steps",                       str(STEPS),
        "--checkpoint_freq",             str(CKPT_FREQ),
        "--save_model_every_checkpoint",
        "--hparams",                     hp,
    ]

    t0 = time.time()

    # Stream output line-by-line for live Colab monitoring
    proc = subprocess.Popen(
        cmd,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in proc.stdout:
        print(line, end="")
    proc.wait()

    elapsed = time.time() - t0
    minutes = elapsed / 60

    if proc.returncode != 0:
        print(f"\n✗ FAILED  ({algo} on {ds})  — return code {proc.returncode}")
        print("  Continuing to next run...\n")
        continue

    print(f"\n✓ DONE  {algo} on {ds}  ({minutes:.1f} min)")

    # ── Persist to Google Drive ──
    drive_dest = os.path.join(DRIVE_OUTPUT, run_tag)
    if os.path.exists(drive_dest):
        shutil.rmtree(drive_dest)
    shutil.copytree(run_outdir, drive_dest)
    print(f"  → Copied to Drive: {drive_dest}")

print("\n" + "=" * 72)
print("  ALL RUNS COMPLETE")
print("=" * 72)


  RUN 1/3:  SpawriousM2M_easy  ×  EQRM
  hparams : {"resnet18": true, "batch_size": 128, "resnet_dropout": 0.1, "eqrm_quantile": 0.75, "eqrm_burnin_iters": 2500, "eqrm_lr": 5e-05}
  output  : /content/domainbed_outputs/SpawriousM2M_easy__EQRM__lr5e-05

Environment:
	Python: 3.12.13
	PyTorch: 2.10.0+cu128
	Torchvision: 0.25.0+cu128
	CUDA: 12.8
	CUDNN: 91002
	NumPy: 2.0.2
	PIL: 11.3.0
Args:
	algorithm: EQRM
	checkpoint_freq: 5001
	data_dir: /content/domainbed_data/spawrious224
	dataset: SpawriousM2M_easy
	holdout_fraction: 0.2
	hparams: {"resnet18": true, "batch_size": 128, "resnet_dropout": 0.1, "eqrm_quantile": 0.75, "eqrm_burnin_iters": 2500, "eqrm_lr": 5e-05}
	hparams_seed: 0
	output_dir: /content/domainbed_outputs/SpawriousM2M_easy__EQRM__lr5e-05
	save_model_every_checkpoint: True
	seed: 0
	skip_model_save: False
	steps: 5001
	task: domain_generalization
	test_envs: [0]
	trial_seed: 0
	uda_holdout_fraction: 0
HParams:
	batch_size: 128
	class_balanced: False
	data_augmentation: True

In [ ]:
import json, itertools

# ═══════════════════════════════════════════════════════════════════
# DATASETS — Spawrious M2M at three difficulty levels
# The benchmark contains 4 dog breeds (Bulldog, Dachshund, Corgi,
# Labrador) with 6 background locations. In the M2M splits, *groups*
# of classes correlate with *groups* of backgrounds, and these
# correlations are reversed in the test environment.
# ═══════════════════════════════════════════════════════════════════
DATASETS = [
    "SpawriousM2M_easy",
    "SpawriousM2M_medium",
    "SpawriousM2M_hard",
]

# ═══════════════════════════════════════════════════════════════════
# ALGORITHMS
#   ERM  — standard empirical risk minimization (baseline)
#   EQRM — empirical quantile risk minimization (targets the
#           worst-quantile loss across environments)
# ═══════════════════════════════════════════════════════════════════
ALGORITHMS = ["EQRM"]

# ═══════════════════════════════════════════════════════════════════
# TRAINING KNOBS
# ═══════════════════════════════════════════════════════════════════
TEST_ENV       = 0       # held-out test environment index
STEPS          = 5001    # DomainBed default (N_STEPS in datasets.py)
CKPT_FREQ      = 5001    # checkpoint at the final step

# ═══════════════════════════════════════════════════════════════════
# HYPERPARAMETERS (passed via --hparams JSON string)
#
# Shared across both algorithms:
#   • resnet18        = true   (matches Spawrious paper Table 3)
#   • batch_size      = 256    (maximize A100 throughput; paper used 128)
#   • resnet_dropout  = 0.1    (paper Sec 5: "We keep dropout (0.1) fixed")
#
# EQRM-specific (Eastwood et al., NeurIPS 2022, Appendix E.3):
#   • eqrm_quantile      = 0.75  (paper default; swept U(0.5, 0.99))
#   • eqrm_burnin_iters  = 2500  (paper default; EQRM runs as plain
#                                  ERM for the first 2500 steps, then
#                                  activates the quantile penalty for
#                                  the remaining 2501 steps.
#                                  NOTE: if you ever drop --steps below
#                                  2500, you MUST lower this too or EQRM
#                                  silently degrades to plain ERM.)
#   • eqrm_lr             = 1e-6  (paper default; lower LR post burn-in
#                                  to stabilize quantile optimization)
# ═══════════════════════════════════════════════════════════════════
SHARED_HPARAMS = {
    "resnet18":       True,
    "batch_size":     128,
    "resnet_dropout": 0.1,
}

ALGO_SPECIFIC_HPARAMS = {
    "ERM":  {},
    "EQRM": {
        "eqrm_quantile":     0.90,
        "eqrm_burnin_iters": 2500,
        "eqrm_lr":           1e-6,
    },
}

# ── Build the full (dataset × algorithm) run manifest ──
runs = []
for ds, algo in itertools.product(DATASETS, ALGORITHMS):
    hparams = {**SHARED_HPARAMS, **ALGO_SPECIFIC_HPARAMS[algo]}
    runs.append({"dataset": ds, "algorithm": algo, "hparams": hparams})

print(f"Total runs: {len(runs)}")
print(f"  {len(DATASETS)} datasets × {len(ALGORITHMS)} algorithms\n")
for i, r in enumerate(runs):
    print(f"  [{i}] {r['dataset']:25s}  {r['algorithm']:5s}  "
          f"hparams={json.dumps(r['hparams'])}")

Total runs: 3
  3 datasets × 1 algorithms

  [0] SpawriousM2M_easy          EQRM   hparams={"resnet18": true, "batch_size": 128, "resnet_dropout": 0.1, "eqrm_quantile": 0.9, "eqrm_burnin_iters": 2500, "eqrm_lr": 1e-06}
  [1] SpawriousM2M_medium        EQRM   hparams={"resnet18": true, "batch_size": 128, "resnet_dropout": 0.1, "eqrm_quantile": 0.9, "eqrm_burnin_iters": 2500, "eqrm_lr": 1e-06}
  [2] SpawriousM2M_hard          EQRM   hparams={"resnet18": true, "batch_size": 128, "resnet_dropout": 0.1, "eqrm_quantile": 0.9, "eqrm_burnin_iters": 2500, "eqrm_lr": 1e-06}


In [ ]:
import subprocess, shutil, json, time, os

# Ensure subprocesses can also import domainbed
env = os.environ.copy()
env["PYTHONPATH"] = DOMAINBED_DIR + ":" + env.get("PYTHONPATH", "")

for idx, run in enumerate(runs):
    ds   = run["dataset"]
    algo = run["algorithm"]
    hp   = json.dumps(run["hparams"])

    # Per-run output directory
    # Per-run output directory
    quantile_label = run["hparams"].get("eqrm_quantile", "default")
    run_tag    = f"{ds}__{algo}__alpha{quantile_label}"
    run_outdir = os.path.join(LOCAL_OUTPUT_DIR, run_tag)
    os.makedirs(run_outdir, exist_ok=True)

    print("\n" + "=" * 72)
    print(f"  RUN {idx + 1}/{len(runs)}:  {ds}  ×  {algo}")
    print(f"  hparams : {hp}")
    print(f"  output  : {run_outdir}")
    print("=" * 72 + "\n")

    cmd = [
        "python", str(TRAIN_SCRIPT),
        "--data_dir",                    SPAWRIOUS_ROOT,
        "--output_dir",                  run_outdir,
        "--dataset",                     ds,
        "--algorithm",                   algo,
        "--test_env",                    str(TEST_ENV),
        "--steps",                       str(STEPS),
        "--checkpoint_freq",             str(CKPT_FREQ),
        "--save_model_every_checkpoint",
        "--hparams",                     hp,
    ]

    t0 = time.time()

    # Stream output line-by-line for live Colab monitoring
    proc = subprocess.Popen(
        cmd,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in proc.stdout:
        print(line, end="")
    proc.wait()

    elapsed = time.time() - t0
    minutes = elapsed / 60

    if proc.returncode != 0:
        print(f"\n✗ FAILED  ({algo} on {ds})  — return code {proc.returncode}")
        print("  Continuing to next run...\n")
        continue

    print(f"\n✓ DONE  {algo} on {ds}  ({minutes:.1f} min)")

    # ── Persist to Google Drive ──
    drive_dest = os.path.join(DRIVE_OUTPUT, run_tag)
    if os.path.exists(drive_dest):
        shutil.rmtree(drive_dest)
    shutil.copytree(run_outdir, drive_dest)
    print(f"  → Copied to Drive: {drive_dest}")

print("\n" + "=" * 72)
print("  ALL RUNS COMPLETE")
print("=" * 72)


  RUN 1/3:  SpawriousM2M_easy  ×  EQRM
  hparams : {"resnet18": true, "batch_size": 128, "resnet_dropout": 0.1, "eqrm_quantile": 0.9, "eqrm_burnin_iters": 2500, "eqrm_lr": 1e-06}
  output  : /content/domainbed_outputs/SpawriousM2M_easy__EQRM__alpha0.9

Environment:
	Python: 3.12.13
	PyTorch: 2.10.0+cu128
	Torchvision: 0.25.0+cu128
	CUDA: 12.8
	CUDNN: 91002
	NumPy: 2.0.2
	PIL: 11.3.0
Args:
	algorithm: EQRM
	checkpoint_freq: 5001
	data_dir: /content/domainbed_data/spawrious224
	dataset: SpawriousM2M_easy
	holdout_fraction: 0.2
	hparams: {"resnet18": true, "batch_size": 128, "resnet_dropout": 0.1, "eqrm_quantile": 0.9, "eqrm_burnin_iters": 2500, "eqrm_lr": 1e-06}
	hparams_seed: 0
	output_dir: /content/domainbed_outputs/SpawriousM2M_easy__EQRM__alpha0.9
	save_model_every_checkpoint: True
	seed: 0
	skip_model_save: False
	steps: 5001
	task: domain_generalization
	test_envs: [0]
	trial_seed: 0
	uda_holdout_fraction: 0
HParams:
	batch_size: 128
	class_balanced: False
	data_augmentation: True